# Multi-Label Chest X-Ray Classifier — NIH ChestX-ray14

Upgrades the original binary (Pneumonia vs Normal) ResNet-50 classifier to:
- **Multi-label** classification across all 14 NIH ChestX-ray14 findings
- **Focal loss** with per-class alpha derived from training-set frequency, to handle severe class imbalance (Hernia ~0.2% vs Infiltration ~18%)
- **Grad-CAM** visualizations so predictions are inspectable, not a black box
- **Gemini API** clinical decision-support layer that turns raw probabilities into a readable note

**Run this in Colab with a GPU runtime** (Runtime → Change runtime type → T4 GPU). It downloads data from Kaggle, so it won't work in a plain CPU/offline environment.

Estimated time: ~10-15 min on the sample subset (5,606 images), ~1-2 hrs/epoch on the full dataset (112,120 images) on a T4.

## 1. Setup

In [ ]:
!git clone https://github.com/pratyu2h/Summer-projects.git
%cd Summer-projects/pneumonia
!pip install -q kaggle google-genai

## 2. Download data (Kaggle API)

You need a Kaggle account + API token (kaggle.com → Account → Create New API Token, downloads `kaggle.json`).

Two dataset options:
- **`nih-chest-xrays/sample`** (~1.2GB, 5,606 images) — fast, good for iterating and for a portfolio demo. **Default below.**
- **`nih-chest-xrays/data`** (~42GB, all 112,120 images) — the real CheXNet-scale dataset, needed if you want to quote NIH's published AUROC numbers as a comparison point. Swap the dataset name in the cell below; budget a few hours + Colab Pro-level disk.

In [ ]:
from google.colab import files
print("Upload your kaggle.json:")
uploaded = files.upload()  # select kaggle.json from your machine

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
DATASET = "nih-chest-xrays/sample"   # swap to "nih-chest-xrays/data" for the full 42GB dataset
DATA_ROOT = "/content/nih_data"

!kaggle datasets download -d {DATASET} -p {DATA_ROOT} --unzip

In [ ]:
# The sample dataset ships images directly under DATA_ROOT/images and a
# sample_labels.csv instead of Data_Entry_2017.csv + train/test list files.
# Normalize it to the layout data.py expects, with a patient-level split
# (by filename prefix, which is the patient ID in NIH's naming convention).
import pandas as pd, os, glob

if DATASET == "nih-chest-xrays/sample":
    df = pd.read_csv(f"{DATA_ROOT}/sample_labels.csv")
    df = df.rename(columns={"Image Index": "Image Index", "Finding Labels": "Finding Labels"})
    df[["Image Index", "Finding Labels"]].to_csv(f"{DATA_ROOT}/Data_Entry_2017.csv", index=False)

    if not os.path.isdir(f"{DATA_ROOT}/images"):
        os.makedirs(f"{DATA_ROOT}/images", exist_ok=True)
        for f in glob.glob(f"{DATA_ROOT}/sample/images/*.png"):
            os.rename(f, f"{DATA_ROOT}/images/{os.path.basename(f)}")

    patient_id = df["Image Index"].str.slice(0, 8)
    unique_patients = patient_id.unique()
    import numpy as np
    rng = np.random.default_rng(0)
    rng.shuffle(unique_patients)
    split_idx = int(0.85 * len(unique_patients))
    train_patients = set(unique_patients[:split_idx])

    train_mask = patient_id.isin(train_patients)
    df.loc[train_mask, "Image Index"].to_csv(f"{DATA_ROOT}/train_val_list.txt", index=False, header=False)
    df.loc[~train_mask, "Image Index"].to_csv(f"{DATA_ROOT}/test_list.txt", index=False, header=False)

print("train images:", sum(1 for _ in open(f"{DATA_ROOT}/train_val_list.txt")))
print("test images:", sum(1 for _ in open(f"{DATA_ROOT}/test_list.txt")))

## 3. Train

In [ ]:
from train import train

model, history = train(
    data_root=DATA_ROOT,
    epochs=10,
    batch_size=32,
    lr=1e-4,
    image_size=224,
    freeze_backbone=True,      # trains just the head first, then unfreezes layer3+ partway through
    checkpoint_path="best_model.pth",
)

In [ ]:
import matplotlib.pyplot as plt

epochs_r = [h["epoch"] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_r, [h["train_loss"] for h in history], label="train")
axes[0].plot(epochs_r, [h["val_loss"] for h in history], label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()

axes[1].plot(epochs_r, [h["train_auc"] for h in history], label="train")
axes[1].plot(epochs_r, [h["val_auc"] for h in history], label="val")
axes[1].set_title("Mean AUROC"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()

## 4. Per-class AUROC on the held-out set

This is the number to quote on your resume/report — compare per-class against the [CheXNet paper](https://arxiv.org/abs/1711.05225)'s reported AUROCs as a sanity check (they trained on the full 112k-image dataset, so a sample-subset model should be somewhat lower, especially on rare classes).

In [ ]:
from train import run_epoch
from data import NIHChestXrayDataset, get_transforms
from losses import MultiLabelFocalLoss, compute_alpha_from_freq
from torch.utils.data import DataLoader
import torch

val_ds = NIHChestXrayDataset(DATA_ROOT, "test_list.txt", transform=get_transforms(False, 224))
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
loss_fn = MultiLabelFocalLoss(alpha=compute_alpha_from_freq(val_ds.labels), gamma=2.0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
_, mean_auc, per_class_auc = run_epoch(model.to(device), val_loader, loss_fn, device, optimizer=None)

print(f"Mean AUROC: {mean_auc:.4f}\n")
for cls, auc in sorted(per_class_auc.items(), key=lambda x: -x[1]):
    print(f"  {cls:20s} {auc:.4f}")

## 5. Grad-CAM: visualize what the model is looking at

In [ ]:
from gradcam import GradCAM, overlay_heatmap
from model import NIH_CLASSES
import numpy as np

cam_engine = GradCAM(model)
model.eval()

images, labels = next(iter(val_loader))
images = images.to(device)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i in range(4):
    img = images[i:i+1]
    probs = model.predict_proba(img)[0]
    top_class_idx = int(probs.argmax())

    heatmap = cam_engine(img, class_idx=top_class_idx)

    # unnormalize for display
    mean = np.array([0.485, 0.456, 0.406]); std = np.array([0.229, 0.224, 0.225])
    disp = img[0].detach().cpu().permute(1, 2, 0).numpy() * std + mean
    blended = overlay_heatmap(disp, heatmap)

    axes[i].imshow(blended)
    axes[i].set_title(f"{NIH_CLASSES[top_class_idx]}\np={probs[top_class_idx]:.2f}", fontsize=10)
    axes[i].axis("off")

cam_engine.close()
plt.tight_layout(); plt.show()

## 6. Gemini clinical decision-support note

Store your key in Colab Secrets (key icon in the left sidebar) as `GEMINI_API_KEY` — don't paste it directly into a cell.

In [ ]:
from google.colab import userdata
import os
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

from gemini_utils import get_clinical_summary

sample_img, _ = next(iter(val_loader))
sample_img = sample_img[:1].to(device)
probs = model.predict_proba(sample_img)[0].tolist()

note = get_clinical_summary(probs, patient_context="Adult patient, routine chest film.")
print(note)

## 7. Push results back

Commits the trained checkpoint's metrics (not the ~90MB weights file itself — see `.gitignore`) and this executed notebook back to the repo.

In [ ]:
import json
with open("training_history.json", "w") as f:
    json.dump(history, f, indent=2)

!git config --global user.email "you@example.com"
!git config --global user.name "your-name"
!git add training_history.json
!git commit -m "Add training history from Colab run"
!git push